In [1]:
# !pip install matplotlib

In [2]:
# %pip install netCDF4

In [3]:
%pip install -U nbformat

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Load OceanEmbed model and processed data
import numpy as np
import pandas as pd
import xarray as xr
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output

In [5]:
import xarray as xr

final_complete = xr.open_dataset(
    "oceanembed_surface_025.nc",
    engine="netcdf4"
)


In [6]:
# Load trained OceanEmbed CNN

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    "oceanembed_patch_cnn_final.pth",
    map_location=device,
    weights_only=False
)

In [7]:
# Recreate and load the trained OceanEmbed CNN

class OceanTemperatureCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(7, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 15, kernel_size=1)
        )

    def forward(self, x):
        return self.network(x)


model_argo = OceanTemperatureCNN().to(device)

model_argo.load_state_dict(
    checkpoint["model_state_dict"]
)

model_argo.eval()

features = checkpoint["features"]
required_depths = np.array(checkpoint["depths"])
patch_size = checkpoint["patch_size"]

print("OceanEmbed CNN loaded successfully.")
print("Features:", features)
print("Depths:", required_depths)
print("Patch size:", patch_size)

OceanEmbed CNN loaded successfully.
Features: ['sst', 'sss', 'ssh', 'current_u', 'current_v', 'wind_u', 'wind_v']
Depths: [   0    5   10   20   30   50   75  100  125  150  200  300  500  700
 1000]
Patch size: 15


In [8]:
# Prepare surface data for OceanEmbed prediction

surface_features = final_complete[
    features
].to_array(
    dim="feature"
).transpose(
    "time",
    "feature",
    "latitude",
    "longitude"
)

surface_scaled = (
    surface_features
    - checkpoint["train_mean"]
) / checkpoint["train_std"]

surface_scaled_values = (
    surface_scaled.values.astype(np.float32)
)

available_dates = pd.to_datetime(
    final_complete.time.values
)

target_lat = final_complete.latitude.values
target_lon = final_complete.longitude.values

print("Surface prediction data prepared successfully.")
print("Shape:", surface_scaled_values.shape)
print("Dates:", available_dates[0].strftime("%Y-%m-%d"),
      "to", available_dates[-1].strftime("%Y-%m-%d"))
print("Latitude:", target_lat.min(), "to", target_lat.max())
print("Longitude:", target_lon.min(), "to", target_lon.max())

Surface prediction data prepared successfully.
Shape: (15, 7, 100, 240)
Dates: 2025-06-01 to 2025-06-15
Latitude: 5.125 to 29.875
Longitude: 45.125 to 104.875


In [9]:
# Test OceanEmbed prediction at one location

def predict_profile(date, latitude, longitude):

    time_idx = int(
        np.abs(
            available_dates - pd.Timestamp(date)
        ).argmin()
    )

    lat_idx = int(
        np.abs(
            target_lat - latitude
        ).argmin()
    )

    lon_idx = int(
        np.abs(
            target_lon - longitude
        ).argmin()
    )

    half = patch_size // 2

    lat_idx = np.clip(
        lat_idx,
        half,
        len(target_lat) - half - 1
    )

    lon_idx = np.clip(
        lon_idx,
        half,
        len(target_lon) - half - 1
    )

    x_patch = surface_scaled_values[
        time_idx,
        :,
        lat_idx-half:lat_idx+half+1,
        lon_idx-half:lon_idx+half+1
    ]

    x_tensor = torch.tensor(
        x_patch,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        prediction = model_argo(x_tensor)

    prediction = prediction.cpu().numpy()[0]

    depth_profile = prediction[
        :,
        half,
        half
    ]

    selected_date = available_dates[time_idx]
    selected_lat = target_lat[lat_idx]
    selected_lon = target_lon[lon_idx]

    return (
        selected_date,
        selected_lat,
        selected_lon,
        depth_profile
    )


test_result = predict_profile(
    "2025-06-10",
    15.5,
    65.5
)

print("Prediction test successful.")

print(
    "Date:",
    test_result[0].strftime("%Y-%m-%d")
)

print(
    "Location:",
    f"{test_result[1]:.3f}°N, "
    f"{test_result[2]:.3f}°E"
)

print(
    "Profile shape:",
    test_result[3].shape
)

print(
    "Temperature range:",
    f"{test_result[3].min():.2f}°C",
    "to",
    f"{test_result[3].max():.2f}°C"
)

Prediction test successful.
Date: 2025-06-10
Location: 15.375°N, 65.375°E
Profile shape: (15,)
Temperature range: 7.95°C to 30.17°C


In [10]:
# Create OceanEmbed dashboard controls

date_dropdown = widgets.Dropdown(
    options=[
        (
            date.strftime("%Y-%m-%d"),
            date
        )
        for date in available_dates
    ],
    value=available_dates[0],
    description="Date:"
)

lat_slider = widgets.FloatSlider(
    value=15.375,
    min=float(target_lat.min()),
    max=float(target_lat.max()),
    step=0.25,
    description="Latitude:",
    continuous_update=False
)

lon_slider = widgets.FloatSlider(
    value=65.375,
    min=float(target_lon.min()),
    max=float(target_lon.max()),
    step=0.25,
    description="Longitude:",
    continuous_update=False
)

print("Dashboard controls created successfully.")

Dashboard controls created successfully.


In [11]:
# %pip install plotly

In [12]:
# Load processed GLORYS reference

glorys_dashboard = xr.open_dataset(
    "oceanembed_glorys_target_025.nc"
)

print("GLORYS reference loaded successfully.")
print(glorys_dashboard)
print("Variables:", list(glorys_dashboard.data_vars))

GLORYS reference loaded successfully.
<xarray.Dataset> Size: 43MB
Dimensions:    (time: 15, depth: 15, latitude: 100, longitude: 240)
Coordinates:
  * time       (time) datetime64[ns] 120B 2025-06-01 2025-06-02 ... 2025-06-15
  * depth      (depth) float32 60B 0.494 5.078 9.573 ... 541.1 643.6 1.062e+03
  * latitude   (latitude) float64 800B 5.125 5.375 5.625 ... 29.38 29.62 29.88
  * longitude  (longitude) float64 2kB 45.12 45.38 45.62 ... 104.4 104.6 104.9
Data variables:
    thetao     (time, depth, latitude, longitude) float64 43MB ...
Variables: ['thetao']


In [13]:
# Prepare GLORYS reference for selected dashboard location

def get_actual_profile(date, latitude, longitude):

    actual = glorys_dashboard["thetao"].sel(
        time=pd.Timestamp(date),
        latitude=latitude,
        longitude=longitude,
        method="nearest"
    )

    actual_profile = actual.values

    return actual_profile

In [14]:
# Test actual GLORYS profile

actual_profile = get_actual_profile(
    "2025-06-10",
    15.375,
    65.375
)

print("Actual GLORYS profile shape:", actual_profile.shape)
print("Actual temperature range:",
      np.nanmin(actual_profile),
      "to",
      np.nanmax(actual_profile))

Actual GLORYS profile shape: (15,)
Actual temperature range: 7.978407715936171 to 30.085424857072468


In [ ]:
# OceanEmbed final dashboard

import plotly.graph_objects as go
from IPython.display import display, clear_output


date_dropdown = widgets.Dropdown(
    options=[
        date.strftime("%Y-%m-%d")
        for date in available_dates
    ],
    value=available_dates[0].strftime("%Y-%m-%d"),
    description="Date:",
    layout=widgets.Layout(width="300px")
)


lat_slider = widgets.FloatSlider(
    value=15.375,
    min=float(target_lat.min()),
    max=float(target_lat.max()),
    step=0.25,
    description="Latitude:",
    continuous_update=False,
    readout_format=".3f",
    layout=widgets.Layout(width="500px")
)


lon_slider = widgets.FloatSlider(
    value=65.375,
    min=float(target_lon.min()),
    max=float(target_lon.max()),
    step=0.25,
    description="Longitude:",
    continuous_update=False,
    readout_format=".3f",
    layout=widgets.Layout(width="500px")
)


generate_button = widgets.Button(
    description="Generate Temperature Profile",
    button_style="primary",
    icon="line-chart"
)


output = widgets.Output()


def generate_profile(_):

    with output:

        clear_output(wait=True)

        selected_date, selected_lat, selected_lon, predicted_profile = predict_profile(
            date_dropdown.value,
            lat_slider.value,
            lon_slider.value
        )


        surface_values = final_complete.sel(
            time=selected_date,
            latitude=selected_lat,
            longitude=selected_lon
        )


        surface_data = np.array([
            float(surface_values["sst"].values),
            float(surface_values["sss"].values),
            float(surface_values["ssh"].values),
            float(surface_values["current_u"].values),
            float(surface_values["current_v"].values),
            float(surface_values["wind_u"].values),
            float(surface_values["wind_v"].values)
        ])


        if not np.isfinite(surface_data).all():

            display(widgets.HTML(
                """
                <div style="padding:12px;">
                    <h3>⚠️ Prediction unavailable</h3>
                    <p>
                    Complete surface observations are not available
                    at the selected location and date.
                    </p>
                </div>
                """
            ))

            return


        actual_profile = glorys_dashboard["thetao"].sel(
            time=selected_date,
            latitude=selected_lat,
            longitude=selected_lon,
            method="nearest"
        ).values


        valid = (
            np.isfinite(predicted_profile)
            & np.isfinite(actual_profile)
        )


        depths = np.asarray(required_depths)[valid]
        predicted = np.asarray(predicted_profile)[valid]
        actual = np.asarray(actual_profile)[valid]


        error = predicted - actual
        absolute_error = np.abs(error)


        rmse = np.sqrt(
            np.mean(error ** 2)
        )

        mae = np.mean(
            absolute_error
        )

        bias = np.mean(error)


        if len(predicted) > 1:

            correlation = np.corrcoef(
                actual,
                predicted
            )[0, 1]

        else:

            correlation = np.nan


        display(widgets.HTML(
            f"""
            <div style="padding:10px 0;">

                <h1>🌊 OceanEmbed</h1>

                <h3>AI-Based Subsurface Ocean Temperature Prediction</h3>

                <p>
                    <b>Date:</b> {selected_date.strftime("%Y-%m-%d")}
                    &nbsp;&nbsp;&nbsp;
                    <b>Location:</b>
                    {selected_lat:.3f}°N,
                    {selected_lon:.3f}°E
                </p>

            </div>
            """
        ))


        # Prediction location map

        fig_map = go.Figure()


        fig_map.add_trace(
            go.Scattergeo(
                lon=[selected_lon],
                lat=[selected_lat],
                mode="markers",
                marker=dict(
                    size=14
                ),
                name="Selected Location",
                text=[
                    f"{selected_lat:.3f}°N, "
                    f"{selected_lon:.3f}°E"
                ],
                hovertemplate="%{text}<extra></extra>"
            )
        )


        fig_map.update_geos(
            projection_type="equirectangular",
            showland=True,
            showcountries=True,
            showcoastlines=True,
            landcolor="lightgray",
            lonaxis=dict(
                range=[45, 105]
            ),
            lataxis=dict(
                range=[5, 30]
            )
        )


        fig_map.update_layout(
            title="Selected Location — North Indian Ocean",
            height=430,
            margin=dict(
                l=10,
                r=10,
                t=50,
                b=10
            )
        )


        fig_map.show()


        # Surface conditions

        display(
            widgets.HTML(
                "<h3>🌡️ Surface Ocean Conditions</h3>"
            )
        )


        surface_table = pd.DataFrame({

            "Parameter": [
                "Sea Surface Temperature",
                "Sea Surface Salinity",
                "Sea Surface Height",
                "Current U",
                "Current V",
                "Wind U",
                "Wind V"
            ],

            "Value": surface_data,

            "Unit": [
                "°C",
                "PSU",
                "m",
                "m/s",
                "m/s",
                "m/s",
                "m/s"
            ]

        })


        display(
            surface_table.style
            .format({
                "Value": "{:.3f}"
            })
            .hide(axis="index")
        )


        # Accuracy metrics

        display(
            widgets.HTML(
                "<h3>📊 Prediction Accuracy</h3>"
            )
        )


        metrics_table = pd.DataFrame({

            "Metric": [
                "RMSE",
                "MAE",
                "Bias",
                "Correlation"
            ],

            "Value": [
                rmse,
                mae,
                bias,
                correlation
            ],

            "Unit": [
                "°C",
                "°C",
                "°C",
                ""
            ]

        })


        display(
            metrics_table.style
            .format({
                "Value": "{:.3f}"
            })
            .hide(axis="index")
        )


        # Actual vs predicted profile

        display(
            widgets.HTML(
                "<h3>🌊 Actual vs Predicted Temperature Profile</h3>"
            )
        )


        fig_profile = go.Figure()


        fig_profile.add_trace(
            go.Scatter(
                x=actual,
                y=depths,
                mode="lines+markers",
                name="Actual — GLORYS",
                marker=dict(
                    size=7
                ),
                line=dict(
                    width=2
                )
            )
        )


        fig_profile.add_trace(
            go.Scatter(
                x=predicted,
                y=depths,
                mode="lines+markers",
                name="Predicted — OceanEmbed",
                marker=dict(
                    size=7
                ),
                line=dict(
                    width=2
                )
            )
        )


        fig_profile.update_yaxes(
            autorange="reversed",
            title="Depth (m)"
        )


        fig_profile.update_xaxes(
            title="Temperature (°C)"
        )


        fig_profile.update_layout(
            title="Actual vs Predicted Subsurface Temperature",
            height=600,
            margin=dict(
                l=70,
                r=20,
                t=60,
                b=60
            ),
            legend=dict(
                orientation="h",
                yanchor="bottom",
                y=1.02,
                xanchor="center",
                x=0.5
            )
        )


        fig_profile.show()


        # Depth-wise comparison

        display(
            widgets.HTML(
                "<h3>📋 Depth-wise Temperature Comparison</h3>"
            )
        )


        comparison_table = pd.DataFrame({

            "Depth (m)": depths,

            "Actual GLORYS (°C)": actual,

            "Predicted OceanEmbed (°C)": predicted,

            "Error (°C)": error,

            "Absolute Error (°C)": absolute_error

        })


        display(
            comparison_table.style
            .format({
                "Actual GLORYS (°C)": "{:.2f}",
                "Predicted OceanEmbed (°C)": "{:.2f}",
                "Error (°C)": "{:+.2f}",
                "Absolute Error (°C)": "{:.2f}"
            })
            .hide(axis="index")
        )


        # Error explanation

        display(
            widgets.HTML(
                """
                <div style="
                    margin-top:15px;
                    padding:12px;
                    border-left:4px solid #2c7be5;
                    background:#f5f8fc;
                ">

                <b>Error = Predicted − Actual</b>

                <br><br>

                Positive error → OceanEmbed predicted
                a higher temperature.

                <br>

                Negative error → OceanEmbed predicted
                a lower temperature.

                </div>
                """
            )
        )


generate_button.on_click(
    generate_profile
)


dashboard = widgets.VBox([

    widgets.HTML(
        """
        <div style="padding:10px 0;">

            <h1>🌊 OceanEmbed</h1>

            <p style="font-size:17px;">
                <b>AI-Based Subsurface Ocean Temperature Prediction</b>
            </p>

            <p>
                Select a date and location to generate
                the 3D subsurface temperature profile.
            </p>

        </div>
        """
    ),

    date_dropdown,

    lat_slider,

    lon_slider,

    generate_button,

    output

])


display(dashboard)


generate_profile(None)